# Une boucle PyTorch lisible et mesurable

**Objectifs observables (75–100 min).** Calculer un gradient à la main puis avec autograd, lire les formes des tensors, écrire la boucle d'entraînement, mesurer la validation à chaque epoch, tracer les deux pertes et recharger un `state_dict`.

**Prérequis :** régression linéaire, MSE et gradient descent. Installer l'option `.[dl]`. Le jeu est synthétique et la leçon s'exécute sur CPU.

## 1. Autograd sur un exemple que l'on peut vérifier

Soit $L(w)=(wx-y)^2$ avec `w=2`, `x=3`, `y=7`. La prédiction vaut 6, la loss vaut 1, et $dL/dw=2(wx-y)x=2(-1)3=-6$. `requires_grad=True` demande à PyTorch de conserver le graphe des opérations ; `backward()` remplit `weight.grad`. Les gradients s'accumulent par défaut : on les remet à zéro avant une nouvelle itération.

In [ ]:
import torch

torch.set_num_threads(1)
torch.manual_seed(5)
weight = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)
loss = (weight * x - 7) ** 2
loss.backward()
print('loss:', loss.item(), '| dloss/dweight:', weight.grad.item())

In [ ]:
manual_gradient = 2 * (2.0 * 3.0 - 7.0) * 3.0
print('Manual / autograd gradient:', manual_gradient, weight.grad.item())
assert abs(manual_gradient - weight.grad.item()) < 1e-7

## 2. Dataset, DataLoader et vrai split de validation

Nous simulons $y=2,5x-0,7+\epsilon$. Chaque ligne a une feature et une cible, donc `(batch,1)` pour les deux tensors. `DataLoader` mélange **train** et forme des mini-lots. La validation est séparée avant l'apprentissage et ne participe ni au gradient ni aux mises à jour. Son rôle est de détecter un écart entre optimisation et généralisation ; ce n'est pas encore un test indépendant final.

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

rng = torch.Generator().manual_seed(8)
features = torch.linspace(-2, 2, 240).unsqueeze(1)
targets = 2.5 * features - 0.7 + 0.15 * torch.randn(features.shape, generator=rng)
permutation = torch.randperm(len(features), generator=rng)
train_indices, validation_indices = permutation[:192], permutation[192:]
train_features, train_targets = features[train_indices], targets[train_indices]
validation_features, validation_targets = features[validation_indices], targets[validation_indices]
train_loader = DataLoader(TensorDataset(train_features, train_targets), batch_size=32, shuffle=True, generator=rng)
print('train:', train_features.shape, '| validation:', validation_features.shape)
print('number of train batches:', len(train_loader))

**Vérification des formes.** La première dimension des tensors générés donne `n=240`. Après split, les datasets train et validation gardent une colonne. Le modèle recevra donc `(batch,1)` et renverra `(batch,1)`.

## 3. Deux boucles explicites

`nn.Linear(1,1)` apprend un poids et un biais. MSE calcule la moyenne des erreurs au carré. Pour un mini-lot : `zero_grad()` → `forward` → `loss` → `backward()` → `step()`. En validation : `eval()` et `inference_mode()` ; aucune mise à jour. Nous enregistrons la moyenne pondérée par le nombre d'exemples, car le dernier lot peut être plus petit.

SGD ajuste $\theta\leftarrow\theta-\eta\nabla L$. Momentum accumule une direction lissée ; Adam adapte les pas par paramètre. Ces variantes changent l'optimisation, pas le protocole de validation.

In [ ]:
model = nn.Linear(in_features=1, out_features=1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.08)

train_loss_history, validation_loss_history = [], []
for epoch in range(80):
    model.train()
    train_loss_sum = 0.0
    for batch_features, batch_targets in train_loader:
        optimizer.zero_grad()
        prediction = model(batch_features)
        loss = loss_fn(prediction, batch_targets)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * len(batch_features)
    train_loss_history.append(train_loss_sum / len(train_features))
    model.eval()
    with torch.inference_mode():
        validation_loss_history.append(loss_fn(model(validation_features), validation_targets).item())
print('weight/bias:', model.weight.item(), model.bias.item())
print('last train/validation MSE:', round(train_loss_history[-1], 4), round(validation_loss_history[-1], 4))

## 4. Voir la progression et les prédictions

Les deux pertes suivent l'optimisation et la validation sur le même jeu générateur. Une perte train faible avec validation qui monte serait un signe d'overfitting. Ici, les courbes proches sont attendues puisque la relation vraie est presque linéaire et le modèle est une droite. La figure de droite compare les points de validation à la droite apprise.

In [ ]:
model.eval()
with torch.inference_mode():
    probe = torch.tensor([[-1.0], [0.0], [1.0]])
    print('x / prediction:', list(zip(probe.ravel().tolist(), model(probe).ravel().tolist())))

In [ ]:
import matplotlib.pyplot as plt

with torch.inference_mode():
    validation_prediction = model(validation_features).numpy().ravel()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(train_loss_history, label='Train MSE')
axes[0].plot(validation_loss_history, label='Validation MSE')
axes[0].set(xlabel='Epoch', ylabel='MSE', title='Apprentissage et validation')
axes[0].legend(); axes[0].grid(alpha=0.2)
axes[1].scatter(validation_features.numpy().ravel(), validation_targets.numpy().ravel(), s=14, alpha=0.6, label='Observations validation')
order = torch.argsort(validation_features.ravel()).numpy()
axes[1].plot(validation_features.numpy().ravel()[order], validation_prediction[order], color='#b4502b', label='Droite apprise')
axes[1].set(xlabel='x', ylabel='y', title='Prédictions sur validation')
axes[1].legend(); plt.tight_layout(); plt.show()

## 5. Checkpoint et device

Le `state_dict` contient les paramètres appris. Pour reproduire une prédiction, il faut également connaître l'architecture, les features et leurs transformations, le code, les versions et la graine. Un modèle et ses lots doivent se trouver sur le même `device`. Pour CUDA, choisir `torch.device('cuda' if torch.cuda.is_available() else 'cpu')` et déplacer modèle **et** batches. Certains calculs GPU restent non déterministes selon la plateforme.

In [ ]:
import io

buffer = io.BytesIO()
torch.save(model.state_dict(), buffer)
buffer.seek(0)
restored = nn.Linear(1, 1)
restored.load_state_dict(torch.load(buffer, weights_only=True))
print('restored prediction:', restored(torch.tensor([[0.0]])).item())

## À retenir et exercices

Le mode `train()` ou `eval()` influence certaines couches comme dropout et batch normalization. `inference_mode()` réduit le travail d'autograd pendant la validation. `state_dict` sauvegarde les poids, pas tout le protocole.

1. **Easy :** écrivez à la main la dérivée du premier exemple scalaire.
2. **Medium :** changez le batch size et comparez le nombre d'updates par epoch, en gardant le split fixe.
3. **Hard :** comparez SGD à Adam sur 3 graines avec la même validation ; documentez ce que vous retenez et créez un test final distinct avant toute revendication de performance.